In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cu121

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

X_train = np.load("X_train.npy")
X_val = np.load("X_val.npy")
y_train = np.load("y_train.npy")
y_val = np.load("y_val.npy")
ocean_mask = np.load("ocean_mask.npy")

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("Device available:", "cuda" if torch.cuda.is_available() else "cpu")

X_train: (145, 76, 76, 3) y_train: (145, 76, 76, 14)
Device available: cuda


In [4]:
!nvidia-smi

Wed Sep  2 20:05:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.86                 Driver Version: 581.86         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8              1W /   80W |     405MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [6]:
!pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [8]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

X_train = np.load("X_train.npy")
X_val = np.load("X_val.npy")
y_train = np.load("y_train.npy")
y_val = np.load("y_val.npy")
ocean_mask = np.load("ocean_mask.npy")

X_train_clean = np.nan_to_num(X_train, nan=0.0)
X_val_clean = np.nan_to_num(X_val, nan=0.0)
y_train_clean = np.nan_to_num(y_train, nan=0.0)
y_val_clean = np.nan_to_num(y_val, nan=0.0)

print("X_train:", X_train_clean.shape, "y_train:", y_train_clean.shape)
print("NaN check:", np.isnan(X_train_clean).sum(), np.isnan(y_train_clean).sum())

X_train: (145, 76, 76, 3) y_train: (145, 76, 76, 14)
NaN check: 0 0


In [9]:
class OceanDataset(Dataset):
    def __init__(self, X, y):
        # Reorder from (N, H, W, C) to (N, C, H, W) — PyTorch conv layers expect channels first
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = OceanDataset(X_train_clean, y_train_clean)
val_dataset = OceanDataset(X_val_clean, y_val_clean)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Sanity check shapes
sample_X, sample_y = next(iter(train_loader))
print("Batch X:", sample_X.shape)  # expect (8, 3, 76, 76)
print("Batch y:", sample_y.shape)  # expect (8, 14, 76, 76)

Batch X: torch.Size([8, 3, 76, 76])
Batch y: torch.Size([8, 14, 76, 76])


In [10]:
class OceanCNN(nn.Module):
    def __init__(self, in_channels=3, out_channels=14):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)  # 76 -> 38
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)  # 38 -> 19

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()
        )

        # Decoder
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)  # 19 -> 38
        self.dec2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),  # 128 = 64(up) + 64(skip)
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)  # 38 -> 76
        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),  # 64 = 32(up) + 32(skip)
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )

        self.out = nn.Conv2d(32, out_channels, 1)  # final 1x1 conv to 14 depth channels

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))  # skip connection, same as your U-Net
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))  # skip connection

        return self.out(d1)

model = OceanCNN(in_channels=3, out_channels=14)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

OceanCNN(
  (enc1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bottleneck): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (up2): ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=(2, 2))
  (dec2): Sequential(
    (0): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
   

In [11]:
# Prepare ocean mask as a tensor, broadcastable across depth channels
ocean_mask_tensor = torch.tensor(ocean_mask, dtype=torch.float32).to(device)  # shape (76, 76)

def masked_mse_loss(pred, target, mask):
    # mask shape (76,76) -> broadcast to (B, C, 76, 76)
    mask_expanded = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, 76, 76)
    squared_error = (pred - target) ** 2
    masked_error = squared_error * mask_expanded
    return masked_error.sum() / (mask_expanded.sum() * pred.shape[0] * pred.shape[1] + 1e-8)

In [12]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 50

train_losses, val_losses = [], []

for epoch in range(num_epochs):
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = masked_mse_loss(pred, y_batch, ocean_mask_tensor)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = masked_mse_loss(pred, y_batch, ocean_mask_tensor)
            epoch_val_loss += loss.item()

    avg_train = epoch_train_loss / len(train_loader)
    avg_val = epoch_val_loss / len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    if epoch % 5 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

Epoch 1/50 | Train Loss: 144.9871 | Val Loss: 35.7079
Epoch 6/50 | Train Loss: 12.4499 | Val Loss: 15.2623
Epoch 11/50 | Train Loss: 7.4556 | Val Loss: 10.0349
Epoch 16/50 | Train Loss: 5.2676 | Val Loss: 8.1897
Epoch 21/50 | Train Loss: 4.2385 | Val Loss: 5.5924
Epoch 26/50 | Train Loss: 3.1313 | Val Loss: 7.0255
Epoch 31/50 | Train Loss: 2.9289 | Val Loss: 4.0177
Epoch 36/50 | Train Loss: 2.5911 | Val Loss: 3.8211
Epoch 41/50 | Train Loss: 2.3486 | Val Loss: 6.4588
Epoch 46/50 | Train Loss: 1.9190 | Val Loss: 4.6523
Epoch 50/50 | Train Loss: 1.8667 | Val Loss: 3.2846


In [13]:
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(50):  # continue from where you are, or restart with this if you want a clean log
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = masked_mse_loss(pred, y_batch, ocean_mask_tensor)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = masked_mse_loss(pred, y_batch, ocean_mask_tensor)
            epoch_val_loss += loss.item()

    avg_train = epoch_train_loss / len(train_loader)
    avg_val = epoch_val_loss / len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")

    if epoch % 5 == 0 or epoch == 49:
        print(f"Epoch {epoch+1}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Best Val: {best_val_loss:.4f}")

Epoch 1/50 | Train: 1.7812 | Val: 4.4422 | Best Val: 4.4422
Epoch 6/50 | Train: 2.2674 | Val: 5.6698 | Best Val: 3.1962
Epoch 11/50 | Train: 1.7053 | Val: 2.9990 | Best Val: 2.9990
Epoch 16/50 | Train: 2.0000 | Val: 3.0772 | Best Val: 2.7268
Epoch 21/50 | Train: 1.9918 | Val: 2.7441 | Best Val: 2.6641
Epoch 26/50 | Train: 1.6149 | Val: 2.6866 | Best Val: 2.6641
Epoch 31/50 | Train: 1.3534 | Val: 4.0882 | Best Val: 2.6641
Epoch 36/50 | Train: 1.8269 | Val: 4.4707 | Best Val: 2.5035
Epoch 41/50 | Train: 1.1971 | Val: 2.6170 | Best Val: 2.5035
Epoch 46/50 | Train: 1.5645 | Val: 2.4501 | Best Val: 2.4501
Epoch 50/50 | Train: 1.7453 | Val: 5.7296 | Best Val: 2.4501


In [14]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
# call scheduler.step(avg_val) after computing avg_val each epoch

In [15]:
for i, name in enumerate(["SST", "SSH", "SSS"]):
    print(f"{name}: mean={X_train_clean[...,i].mean():.4f}  std={X_train_clean[...,i].std():.4f}")

SST: mean=202.3233  std=141.8839
SSH: mean=0.0257  std=0.0817
SSS: mean=18.7447  std=16.2756


In [16]:
import numpy as np

X_train_raw = np.load("X_train.npy")
X_val_raw   = np.load("X_val.npy")
ocean_mask  = np.load("ocean_mask.npy")

mask = ocean_mask.astype(bool)
if mask.ndim == 3:
    mask = mask[0]

# Per-channel stats over OCEAN pixels only, from TRAIN split only (no val leakage)
means, stds = [], []
for c in range(X_train_raw.shape[-1]):
    vals = X_train_raw[..., c][:, mask]
    vals = vals[~np.isnan(vals)]
    means.append(vals.mean())
    stds.append(vals.std())
means = np.array(means, dtype=np.float32)
stds  = np.array(stds,  dtype=np.float32)
print("ocean-only mean:", means)
print("ocean-only std :", stds)

def prep(X):
    Z = (X - means) / stds      # standardize FIRST
    Z[:, ~mask, :] = 0.0        # THEN neutralize land
    return np.nan_to_num(Z, nan=0.0).astype(np.float32)

X_train_clean = prep(X_train_raw)
X_val_clean   = prep(X_val_raw)

np.save("input_scaler.npy", np.stack([means, stds]))   # needed at inference

for i, name in enumerate(["SST", "SSH", "SSS"]):
    v = X_train_clean[..., i][:, mask]
    print(f"{name}: mean={v.mean():.4f}  std={v.std():.4f}")

ocean-only mean: [3.0181287e+02 3.8412638e-02 3.2860214e+01]
ocean-only std : [1.7205828  0.09734134 0.729624  ]
SST: mean=-0.0000  std=1.0000
SSH: mean=-0.0000  std=0.9995
SSS: mean=-0.0000  std=0.9225


In [17]:
# Per-depth climatology baseline — "just predict the training mean at each depth"
clim = y_train_clean[:, mask, :].mean(axis=(0, 1))        # (14,)
err  = y_val_clean[:, mask, :] - clim                      # (n_val, n_ocean, 14)

rmse_clim       = np.sqrt((err**2).mean())
rmse_clim_depth = np.sqrt((err**2).mean(axis=(0, 1)))      # (14,)

print(f"Climatology RMSE (all depths): {rmse_clim:.3f}")
print("Per-depth:", np.round(rmse_clim_depth, 3))

Climatology RMSE (all depths): 6.658
Per-depth: [3.738 3.755 6.141 7.432 8.332 8.989 9.009 8.627 7.772 6.928 5.705 4.711
 4.159 3.657]


In [18]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch).cpu().numpy()
        all_preds.append(pred)
        all_true.append(y_batch.numpy())

preds = np.concatenate(all_preds, axis=0).transpose(0, 2, 3, 1)  # (N, H, W, 14)
trues = np.concatenate(all_true, axis=0).transpose(0, 2, 3, 1)

err_model = trues[:, mask, :] - preds[:, mask, :]
rmse_model_depth = np.sqrt((err_model**2).mean(axis=(0, 1)))

print("Model per-depth RMSE:", np.round(rmse_model_depth, 3))
print("Climatology per-depth RMSE:", np.round(rmse_clim_depth, 3))
print("Improvement factor:", np.round(rmse_clim_depth / rmse_model_depth, 2))

Model per-depth RMSE: [1.096 1.13  1.239 1.152 1.175 1.662 2.4   2.463 2.274 1.945 1.268 0.91
 0.83  0.84 ]
Climatology per-depth RMSE: [3.738 3.755 6.141 7.432 8.332 8.989 9.009 8.627 7.772 6.928 5.705 4.711
 4.159 3.657]
Improvement factor: [3.41 3.32 4.96 6.45 7.09 5.41 3.75 3.5  3.42 3.56 4.5  5.18 5.01 4.35]


In [19]:
import os
import numpy as np
import torch

print("=== FILES IN CURRENT DIRECTORY ===")
for f in sorted(os.listdir(".")):
    if f.endswith((".npy", ".pth", ".nc")):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f:30s} {size_kb:10.1f} KB")

print("\n=== DATA ARRAYS ===")
for fname in ["X_train.npy", "X_val.npy", "y_train.npy", "y_val.npy",
              "ocean_mask.npy", "input_scaler.npy",
              "val_predictions.npy", "ocean_mask_bool.npy"]:
    if os.path.exists(fname):
        arr = np.load(fname)
        nan_count = np.isnan(arr).sum() if np.issubdtype(arr.dtype, np.floating) else "N/A (bool)"
        print(f"{fname:25s} shape={str(arr.shape):20s} dtype={arr.dtype} nan_count={nan_count}")
    else:
        print(f"{fname:25s} MISSING")

print("\n=== MODEL CHECKPOINTS ===")
for fname in ["best_model.pth", "best_model_baseline.pth"]:
    if os.path.exists(fname):
        sd = torch.load(fname, map_location="cpu")
        n_params = sum(v.numel() for v in sd.values())
        mtime = os.path.getmtime(fname)
        import datetime
        print(f"{fname:25s} exists, {n_params:,} params, last modified: {datetime.datetime.fromtimestamp(mtime)}")
    else:
        print(f"{fname:25s} MISSING")

print("\n=== ARE best_model.pth AND best_model_baseline.pth DIFFERENT? ===")
if os.path.exists("best_model.pth") and os.path.exists("best_model_baseline.pth"):
    sd1 = torch.load("best_model.pth", map_location="cpu")
    sd2 = torch.load("best_model_baseline.pth", map_location="cpu")
    same = all(torch.equal(sd1[k], sd2[k]) for k in sd1.keys())
    print("IDENTICAL weights" if same else "DIFFERENT weights (good — confirms separate checkpoints)")
else:
    print("Can't compare — one or both files missing")

print("\n=== INPUT SCALER SANITY ===")
if os.path.exists("input_scaler.npy"):
    scaler = np.load("input_scaler.npy")
    means, stds = scaler[0], scaler[1]
    print(f"means: {means}")
    print(f"stds:  {stds}")

print("\n=== CLIMATOLOGY BASELINE (run fresh, not from memory) ===")
if os.path.exists("y_train.npy") and os.path.exists("y_val.npy") and os.path.exists("ocean_mask.npy"):
    y_train_raw = np.load("y_train.npy")
    y_val_raw = np.load("y_val.npy")
    ocean_mask_raw = np.load("ocean_mask.npy")
    mask = ocean_mask_raw.astype(bool)
    if mask.ndim == 3:
        mask = mask[0]
    y_train_clean = np.nan_to_num(y_train_raw, nan=0.0)
    y_val_clean = np.nan_to_num(y_val_raw, nan=0.0)

    clim = y_train_clean[:, mask, :].mean(axis=0)  # (n_ocean, 14)
    err = y_val_clean[:, mask, :] - clim
    rmse_clim = np.sqrt((err**2).mean())
    rmse_clim_depth = np.sqrt((err**2).mean(axis=0))
    print(f"Climatology RMSE (all depths): {rmse_clim:.3f}")
    print(f"Per-depth RMSE: {np.round(rmse_clim_depth, 3)}")
else:
    print("Missing y_train/y_val/mask — can't compute")

=== FILES IN CURRENT DIRECTORY ===
  X_train.npy                       19629.5 KB
  X_val.npy                          5009.0 KB
  best_model.pth                     1833.2 KB
  glorys_test.nc                    29697.9 KB
  glorys_test_(1).nc                29697.9 KB
  glorys_train.nc                  722635.6 KB
  input_scaler.npy                      0.1 KB
  ocean_mask.npy                        5.8 KB
  ocean_mask_bool.npy                   5.8 KB
  ssh_test.nc                         777.8 KB
  ssh_train.nc                      18225.1 KB
  sss_test.nc                         250.1 KB
  sss_train.nc                       1606.1 KB
  sst_test.nc                        2375.4 KB
  sst_train.nc                      56904.7 KB
  val_predictions.npy               11687.5 KB
  y_train.npy                       91603.9 KB
  y_val.npy                         23374.9 KB

=== DATA ARRAYS ===
X_train.npy               shape=(145, 76, 76, 3)     dtype=float64 nan_count=912506
X_val.npy     

In [20]:
import numpy as np

# Load everything fresh from disk
preds = np.load("val_predictions.npy")          # (37, 76, 76, 14) — from CURRENT best_model.pth
y_val_raw = np.load("y_val.npy")
ocean_mask_raw = np.load("ocean_mask.npy")

mask = ocean_mask_raw.astype(bool)
if mask.ndim == 3:
    mask = mask[0]

y_val_clean = np.nan_to_num(y_val_raw, nan=0.0)

# --- Model's actual validation RMSE (masked to ocean pixels only) ---
model_err = preds[:, mask, :] - y_val_clean[:, mask, :]     # (37, n_ocean, 14)
rmse_model_overall = np.sqrt((model_err**2).mean())
rmse_model_depth = np.sqrt((model_err**2).mean(axis=(0, 1)))  # FIXED: average over days AND pixels

# --- Climatology baseline, corrected ---
y_train_raw = np.load("y_train.npy")
y_train_clean = np.nan_to_num(y_train_raw, nan=0.0)
clim = y_train_clean[:, mask, :].mean(axis=0)                # (n_ocean, 14)
clim_err = y_val_clean[:, mask, :] - clim
rmse_clim_overall = np.sqrt((clim_err**2).mean())
rmse_clim_depth = np.sqrt((clim_err**2).mean(axis=(0, 1)))    # FIXED

print(f"MODEL       overall RMSE: {rmse_model_overall:.3f}")
print(f"CLIMATOLOGY overall RMSE: {rmse_clim_overall:.3f}")
print()
print(f"{'Depth idx':>10} {'Model RMSE':>12} {'Clim RMSE':>12} {'Model beats clim?':>20}")
for i in range(14):
    beats = "YES" if rmse_model_depth[i] < rmse_clim_depth[i] else "no"
    print(f"{i:>10} {rmse_model_depth[i]:>12.3f} {rmse_clim_depth[i]:>12.3f} {beats:>20}")

MODEL       overall RMSE: 19.666
CLIMATOLOGY overall RMSE: 1.872

 Depth idx   Model RMSE    Clim RMSE    Model beats clim?
         0       24.267        1.603                   no
         1       24.320        1.654                   no
         2       24.239        1.574                   no
         3       24.704        1.589                   no
         4       25.464        1.739                   no
         5       23.881        2.180                   no
         6       22.350        2.801                   no
         7       19.823        2.944                   no
         8       17.196        2.708                   no
         9       14.919        2.252                   no
        10       12.694        1.280                   no
        11        9.947        0.527                   no
        12        9.074        0.268                   no
        13        8.025        0.271                   no


In [21]:
import numpy as np
import torch

preds = np.load("val_predictions.npy")
y_val_raw = np.load("y_val.npy")
ocean_mask_raw = np.load("ocean_mask.npy")
mask = ocean_mask_raw.astype(bool)
if mask.ndim == 3:
    mask = mask[0]
y_val_clean = np.nan_to_num(y_val_raw, nan=0.0)

print("=== PREDICTION STATS (ocean pixels only) ===")
p = preds[:, mask, :]
print(f"preds  mean={p.mean():.3f}  std={p.std():.3f}  min={p.min():.3f}  max={p.max():.3f}")

print("\n=== TRUE VALUE STATS (ocean pixels only) ===")
y = y_val_clean[:, mask, :]
print(f"y_val  mean={y.mean():.3f}  std={y.std():.3f}  min={y.min():.3f}  max={y.max():.3f}")

print("\n=== Is the model just predicting near-zero / near-constant? ===")
print(f"preds std across all values: {p.std():.4f}  (if this is tiny, model collapsed)")
print(f"preds std across depth-means: {p.mean(axis=(0,1)).std():.4f}")

print("\n=== CHECKPOINT LOAD SANITY ===")
sd = torch.load("best_model.pth", map_location="cpu")
# print a couple of weight stats — near-zero everywhere with tiny std = suspicious
for k in list(sd.keys())[:3]:
    w = sd[k]
    print(f"{k:30s} mean={w.mean().item():.5f} std={w.std().item():.5f}")

=== PREDICTION STATS (ocean pixels only) ===
preds  mean=2.504  std=3.129  min=-6.066  max=21.014

=== TRUE VALUE STATS (ocean pixels only) ===
y_val  mean=19.838  std=10.553  min=0.000  max=33.304

=== Is the model just predicting near-zero / near-constant? ===
preds std across all values: 3.1285  (if this is tiny, model collapsed)
preds std across depth-means: 2.1236

=== CHECKPOINT LOAD SANITY ===
enc1.0.weight                  mean=0.06024 std=0.47898
enc1.0.bias                    mean=-0.02817 std=0.11081
enc1.2.weight                  mean=-0.00133 std=0.06512


In [22]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ---- Reload standardized data fresh ----
X_train_raw = np.load("X_train.npy")
X_val_raw   = np.load("X_val.npy")
y_train_raw = np.load("y_train.npy")
y_val_raw   = np.load("y_val.npy")
ocean_mask  = np.load("ocean_mask.npy")

mask = ocean_mask.astype(bool)
if mask.ndim == 3:
    mask = mask[0]

scaler = np.load("input_scaler.npy")
means, stds = scaler[0], scaler[1]

def prep(X):
    Z = (X - means) / stds
    Z[:, ~mask, :] = 0.0
    return np.nan_to_num(Z, nan=0.0).astype(np.float32)

X_train_clean = prep(X_train_raw)
X_val_clean   = prep(X_val_raw)
y_train_clean = np.nan_to_num(y_train_raw, nan=0.0).astype(np.float32)
y_val_clean   = np.nan_to_num(y_val_raw, nan=0.0).astype(np.float32)

class OceanDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_dataset = OceanDataset(X_train_clean, y_train_clean)
val_dataset   = OceanDataset(X_val_clean, y_val_clean)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=8, shuffle=False)

class OceanCNN(nn.Module):
    def __init__(self, in_channels=3, out_channels=14):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.Conv2d(128, 128, 3, padding=1), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(32, out_channels, 1)
    def forward(self, x):
        e1 = self.enc1(x); p1 = self.pool1(e1)
        e2 = self.enc2(p1); p2 = self.pool2(e2)
        b = self.bottleneck(p2)
        u2 = self.up2(b); d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2); d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return self.out(d1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = OceanCNN(in_channels=3, out_channels=14).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

mask_tensor = torch.tensor(mask, dtype=torch.bool, device=device)

def masked_mse_loss(pred, target, mask_2d):
    m = mask_2d.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    diff2 = (pred - target) ** 2
    diff2 = diff2 * m
    return diff2.sum() / (m.sum() * pred.shape[0] * pred.shape[1] + 1e-8)

best_val_loss = float('inf')
print(f"Training on: {device}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

for epoch in range(50):
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = masked_mse_loss(pred, y_batch, mask_tensor)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = masked_mse_loss(pred, y_batch, mask_tensor)
            epoch_val_loss += loss.item()

    avg_train = epoch_train_loss / len(train_loader)
    avg_val = epoch_val_loss / len(val_loader)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")

    # print EVERY epoch this time — no silent gaps
    print(f"Epoch {epoch+1:2d}/50 | Train: {avg_train:7.4f} | Val: {avg_val:7.4f} | Best: {best_val_loss:7.4f}")

print("\nDone. best_model.pth overwritten with this run's best checkpoint.")

Training on: cuda
Train batches: 19, Val batches: 5
Epoch  1/50 | Train: 287.5261 | Val: 93.1247 | Best: 93.1247
Epoch  2/50 | Train: 61.1436 | Val: 74.5087 | Best: 74.5087
Epoch  3/50 | Train: 40.6680 | Val: 52.5011 | Best: 52.5011
Epoch  4/50 | Train: 40.2190 | Val: 41.7665 | Best: 41.7665
Epoch  5/50 | Train: 38.3400 | Val: 39.8934 | Best: 39.8934
Epoch  6/50 | Train: 33.1045 | Val: 41.6460 | Best: 39.8934
Epoch  7/50 | Train: 35.4724 | Val: 44.1604 | Best: 39.8934
Epoch  8/50 | Train: 30.8786 | Val: 36.3178 | Best: 36.3178
Epoch  9/50 | Train: 28.3732 | Val: 35.5490 | Best: 35.5490
Epoch 10/50 | Train: 26.8952 | Val: 34.2431 | Best: 34.2431
Epoch 11/50 | Train: 25.2256 | Val: 33.1056 | Best: 33.1056
Epoch 12/50 | Train: 24.5787 | Val: 32.3787 | Best: 32.3787
Epoch 13/50 | Train: 23.1726 | Val: 30.7240 | Best: 30.7240
Epoch 14/50 | Train: 22.6043 | Val: 38.3356 | Best: 30.7240
Epoch 15/50 | Train: 22.7675 | Val: 28.9222 | Best: 28.9222
Epoch 16/50 | Train: 20.9733 | Val: 27.4721 | B

In [23]:
# Continue from current best_model.pth, same optimizer state ideally,
# but simplest: just run another 50 epochs appended to the same loop logic,
# reusing model/optimizer/train_loader/val_loader already in memory
for epoch in range(50, 100):
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = masked_mse_loss(pred, y_batch, mask_tensor)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = masked_mse_loss(pred, y_batch, mask_tensor)
            epoch_val_loss += loss.item()

    avg_train = epoch_train_loss / len(train_loader)
    avg_val = epoch_val_loss / len(val_loader)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch+1:3d}/100 | Train: {avg_train:7.4f} | Val: {avg_val:7.4f} | Best: {best_val_loss:7.4f}")

print("\nDone.")

Epoch  51/100 | Train:  7.6756 | Val: 12.0211 | Best: 12.0211
Epoch  52/100 | Train:  7.3689 | Val: 11.2919 | Best: 11.2919
Epoch  53/100 | Train:  7.1455 | Val: 11.2832 | Best: 11.2832
Epoch  54/100 | Train:  7.0286 | Val: 11.5127 | Best: 11.2832
Epoch  55/100 | Train:  6.8609 | Val: 10.8296 | Best: 10.8296
Epoch  56/100 | Train:  6.6421 | Val: 11.1377 | Best: 10.8296
Epoch  57/100 | Train:  6.7747 | Val: 11.7518 | Best: 10.8296
Epoch  58/100 | Train:  6.8101 | Val: 10.4570 | Best: 10.4570
Epoch  59/100 | Train:  6.3990 | Val: 11.0891 | Best: 10.4570
Epoch  60/100 | Train:  6.3377 | Val: 10.4482 | Best: 10.4482
Epoch  61/100 | Train:  6.2717 | Val: 10.4531 | Best: 10.4482
Epoch  62/100 | Train:  6.1446 | Val: 10.9177 | Best: 10.4482
Epoch  63/100 | Train:  5.9182 | Val: 10.3365 | Best: 10.3365
Epoch  64/100 | Train:  5.7764 | Val:  9.8811 | Best:  9.8811
Epoch  65/100 | Train:  6.1309 | Val:  9.8430 | Best:  9.8430
Epoch  66/100 | Train:  6.4701 | Val: 10.9775 | Best:  9.8430
Epoch  6

In [24]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

for epoch in range(100, 175):
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = masked_mse_loss(pred, y_batch, mask_tensor)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            loss = masked_mse_loss(pred, y_batch, mask_tensor)
            epoch_val_loss += loss.item()

    avg_train = epoch_train_loss / len(train_loader)
    avg_val = epoch_val_loss / len(val_loader)

    scheduler.step(avg_val)
    current_lr = optimizer.param_groups[0]['lr']

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch+1:3d}/175 | Train: {avg_train:7.4f} | Val: {avg_val:7.4f} | Best: {best_val_loss:7.4f} | LR: {current_lr:.6f}")

print("\nDone.")

Epoch 101/175 | Train:  4.0610 | Val:  7.6460 | Best:  7.3753 | LR: 0.001000
Epoch 102/175 | Train:  3.6903 | Val:  7.3556 | Best:  7.3556 | LR: 0.001000
Epoch 103/175 | Train:  3.5994 | Val:  7.0384 | Best:  7.0384 | LR: 0.001000
Epoch 104/175 | Train:  3.5278 | Val:  7.0176 | Best:  7.0176 | LR: 0.001000
Epoch 105/175 | Train:  3.5456 | Val:  7.3246 | Best:  7.0176 | LR: 0.001000
Epoch 106/175 | Train:  3.5755 | Val:  6.9114 | Best:  6.9114 | LR: 0.001000
Epoch 107/175 | Train:  3.5565 | Val:  6.9587 | Best:  6.9114 | LR: 0.001000
Epoch 108/175 | Train:  3.4456 | Val:  7.0829 | Best:  6.9114 | LR: 0.001000
Epoch 109/175 | Train:  3.6634 | Val:  7.4098 | Best:  6.9114 | LR: 0.001000
Epoch 110/175 | Train:  3.5118 | Val:  6.7358 | Best:  6.7358 | LR: 0.001000
Epoch 111/175 | Train:  3.3760 | Val:  6.8884 | Best:  6.7358 | LR: 0.001000
Epoch 112/175 | Train:  3.3563 | Val:  6.5984 | Best:  6.5984 | LR: 0.001000
Epoch 113/175 | Train:  3.3395 | Val:  6.6577 | Best:  6.5984 | LR: 0.001000

In [25]:
print("=" * 60)
print("NORMALIZATION AUDIT")
print("=" * 60)

means, stds = np.load("input_scaler.npy")

print("Saved means:")
print(means)

print("\nSaved stds:")
print(stds)


def prep(X):
    Z = (X - means) / stds
    Z[:, ~mask, :] = 0.0
    return np.nan_to_num(Z, nan=0.0).astype(np.float32)


Xtr_norm = prep(Xtr)
Xv_norm  = prep(Xv)


print("\nNORMALIZED TRAIN STATS:")

for i, name in enumerate(channel_names):

    vals = Xtr_norm[..., i][:, mask]

    print(
        f"{name}: "
        f"mean={vals.mean():.6f}, "
        f"std={vals.std():.6f}, "
        f"min={vals.min():.6f}, "
        f"max={vals.max():.6f}"
    )


print("\nLAND CHECK:")

land_values = Xtr_norm[:, ~mask, :]

print(
    "Maximum absolute land input:",
    np.abs(land_values).max()
)

print("\n" + "=" * 60)

NORMALIZATION AUDIT
Saved means:
[3.0181287e+02 3.8412638e-02 3.2860214e+01]

Saved stds:
[1.7205828  0.09734134 0.729624  ]


NameError: name 'Xtr' is not defined

In [26]:
print("=" * 60)
print("CLIMATOLOGY BASELINE")
print("=" * 60)

ytr_clean = np.nan_to_num(ytr, nan=0.0)
yv_clean  = np.nan_to_num(yv,  nan=0.0)

# Mean temperature at each depth over ocean
clim = ytr_clean[:, mask, :].mean(axis=(0, 1))

# Validation error
err = yv_clean[:, mask, :] - clim

rmse_clim = np.sqrt(np.mean(err ** 2))

rmse_depth = np.sqrt(
    np.mean(err ** 2, axis=(0, 1))
)

print(f"\nOverall climatology RMSE: {rmse_clim:.4f} °C")

print("\nPer-depth climatology RMSE:")

for depth, rmse in zip(depths, rmse_depth):
    print(f"{depth:6.1f} m : {rmse:.4f} °C")

print("\n" + "=" * 60)

CLIMATOLOGY BASELINE


NameError: name 'ytr' is not defined

In [27]:
import os
from datetime import datetime

print("=" * 60)
print("CHECKPOINT AUDIT")
print("=" * 60)

path = "best_model.pth"

if os.path.exists(path):

    size_mb = os.path.getsize(path) / (1024**2)
    modified = datetime.fromtimestamp(
        os.path.getmtime(path)
    )

    print("File exists       : YES")
    print(f"Size              : {size_mb:.2f} MB")
    print(f"Last modified     : {modified}")

else:

    print("best_model.pth NOT FOUND")

print("=" * 60)

CHECKPOINT AUDIT
File exists       : YES
Size              : 1.79 MB
Last modified     : 2026-09-02 21:25:35.196198


In [28]:
if os.path.exists("val_predictions.npy"):

    preds = np.load("val_predictions.npy")

    print("=" * 60)
    print("PREDICTION AUDIT")
    print("=" * 60)

    print("Prediction shape:", preds.shape)

    print("\nPrediction statistics:")
    print("Min :", np.nanmin(preds))
    print("Max :", np.nanmax(preds))
    print("Mean:", np.nanmean(preds))
    print("Std :", np.nanstd(preds))

    print("\nTrue validation statistics:")
    print("Min :", np.nanmin(yv))
    print("Max :", np.nanmax(yv))
    print("Mean:", np.nanmean(yv))
    print("Std :", np.nanstd(yv))

    print("\nExpected shape:")
    print("(37, 76, 76, 14)")

else:

    print("val_predictions.npy does not exist.")

PREDICTION AUDIT
Prediction shape: (37, 76, 76, 14)

Prediction statistics:
Min : -6.0658517
Max : 21.014355
Mean: 1.771738
Std : 2.8126578

True validation statistics:


NameError: name 'yv' is not defined

In [29]:
import numpy as np

X_train_raw = np.load("X_train.npy")
X_val_raw   = np.load("X_val.npy")

y_train_raw = np.load("y_train.npy")
y_val_raw   = np.load("y_val.npy")

ocean_mask = np.load("ocean_mask.npy").astype(bool)

if ocean_mask.ndim == 3:
    ocean_mask = ocean_mask[0]

depths = np.array([
    0.5, 5, 10, 20, 30, 50, 75,
    100, 125, 150, 200, 300, 500, 700
])

print("=" * 60)
print("TARGET VALIDITY CHECK")
print("=" * 60)

for i, depth in enumerate(depths):

    train_vals = y_train_raw[..., i][:, ocean_mask]
    val_vals   = y_val_raw[..., i][:, ocean_mask]

    train_valid = np.isfinite(train_vals)
    val_valid   = np.isfinite(val_vals)

    print(
        f"{depth:6.1f} m | "
        f"Train valid: {train_valid.mean():.4%} | "
        f"Val valid: {val_valid.mean():.4%}"
    )

print("\nOverall valid target fraction:")

train_valid = np.isfinite(
    y_train_raw[:, ocean_mask, :]
)

val_valid = np.isfinite(
    y_val_raw[:, ocean_mask, :]
)

print("Train:", train_valid.mean())
print("Val  :", val_valid.mean())

print("=" * 60)

TARGET VALIDITY CHECK
   0.5 m | Train valid: 98.6312% | Val valid: 98.6312%
   5.0 m | Train valid: 98.6312% | Val valid: 98.6312%
  10.0 m | Train valid: 95.8161% | Val valid: 95.8161%
  20.0 m | Train valid: 93.6209% | Val valid: 93.6209%
  30.0 m | Train valid: 91.7355% | Val valid: 91.7355%
  50.0 m | Train valid: 89.3337% | Val valid: 89.3337%
  75.0 m | Train valid: 86.6736% | Val valid: 86.6736%
 100.0 m | Train valid: 84.0909% | Val valid: 84.0909%
 125.0 m | Train valid: 83.3678% | Val valid: 83.3678%
 150.0 m | Train valid: 82.9029% | Val valid: 82.9029%
 200.0 m | Train valid: 81.8698% | Val valid: 81.8698%
 300.0 m | Train valid: 80.8368% | Val valid: 80.8368%
 500.0 m | Train valid: 78.4091% | Val valid: 78.4091%
 700.0 m | Train valid: 76.7562% | Val valid: 76.7562%

Overall valid target fraction:
Train: 0.8733397284533648
Val  : 0.8733397284533648


In [30]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math

X_train = np.load("X_train.npy")
X_val = np.load("X_val.npy")
y_train = np.load("y_train.npy")
y_val = np.load("y_val.npy")
ocean_mask = np.load("ocean_mask.npy")  # 2D, kept for reference/inputs only

print("Raw shapes:", X_train.shape, y_train.shape, X_val.shape, y_val.shape)
print("Raw y_train NaN count:", np.isnan(y_train).sum())

Raw shapes: (145, 76, 76, 3) (145, 76, 76, 14) (37, 76, 76, 3) (37, 76, 76, 14)
Raw y_train NaN count: 4860690


In [31]:
# 4D target validity mask: True where GLORYS actually has data at that (sample, lat, lon, depth)
target_valid_train = ~np.isnan(y_train)   # shape (145, 76, 76, 14)
target_valid_val = ~np.isnan(y_val)       # shape (37, 76, 76, 14)

print("Train valid fraction:", target_valid_train.mean())
print("Val valid fraction:", target_valid_val.mean())

# Fill NaN with 0 ONLY for tensor stability — these positions will be excluded via the mask, never trusted as real
X_train_clean = np.nan_to_num(X_train, nan=0.0)
X_val_clean = np.nan_to_num(X_val, nan=0.0)
y_train_clean = np.nan_to_num(y_train, nan=0.0)
y_val_clean = np.nan_to_num(y_val, nan=0.0)

Train valid fraction: 0.5854521171349426
Val valid fraction: 0.5854521171349426


In [32]:
class OceanDataset(Dataset):
    def __init__(self, X, y, valid_mask):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2)          # (N, 3, 76, 76)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)          # (N, 14, 76, 76)
        self.mask = torch.tensor(valid_mask, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 14, 76, 76)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.mask[idx]

train_dataset = OceanDataset(X_train_clean, y_train_clean, target_valid_train)
val_dataset = OceanDataset(X_val_clean, y_val_clean, target_valid_val)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

sample_X, sample_y, sample_mask = next(iter(train_loader))
print("X:", sample_X.shape, "y:", sample_y.shape, "mask:", sample_mask.shape)

X: torch.Size([8, 3, 76, 76]) y: torch.Size([8, 14, 76, 76]) mask: torch.Size([8, 14, 76, 76])


In [33]:
def masked_mse_loss(pred, target, valid_mask):
    squared_error = (pred - target) ** 2
    masked_error = squared_error * valid_mask
    return masked_error.sum() / (valid_mask.sum() + 1e-8)

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Per-depth mean temperature computed ONLY from valid training pixels
climatology = np.zeros(14)
for d in range(14):
    valid_vals = y_train[..., d][target_valid_train[..., d]]
    climatology[d] = valid_vals.mean()

print("Climatology per depth (°C):", climatology)

# Evaluate climatology baseline on validation set using the SAME masked loss logic
clim_tensor = torch.tensor(climatology, dtype=torch.float32).view(1, 14, 1, 1)
total_sq_error, total_valid = 0.0, 0.0
for X_b, y_b, mask_b in val_loader:
    pred_clim = clim_tensor.expand(y_b.shape[0], 14, 76, 76)
    sq_err = ((pred_clim - y_b) ** 2) * mask_b
    total_sq_error += sq_err.sum().item()
    total_valid += mask_b.sum().item()

clim_mse = total_sq_error / total_valid
print(f"Climatology baseline MSE: {clim_mse:.4f} | RMSE: {math.sqrt(clim_mse):.4f} °C")

Climatology per depth (°C): [28.70641207 28.60552241 28.6208668  28.55062155 28.30579684 27.26493244
 24.92968973 21.81115768 18.92890412 16.68960949 14.01472118 11.71640207
  9.98925591  8.53326547]
Climatology baseline MSE: 2.6223 | RMSE: 1.6194 °C


In [35]:
class OceanCNN(nn.Module):
    def __init__(self, in_channels=3, out_channels=14):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()
        )
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.out = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        b = self.bottleneck(p2)
        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return self.out(d1)

model = OceanCNN(in_channels=3, out_channels=14).to(device)
CHECKPOINT_NAME = "baseline_maskfix_v1.pth"  # new name — never overwrites best_model.pth
print(f"Model ready. Will save to: {CHECKPOINT_NAME}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Model ready. Will save to: baseline_maskfix_v1.pth
Params: 466,958


In [36]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 50
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    model.train()
    epoch_train_loss = 0
    for X_b, y_b, mask_b in train_loader:
        X_b, y_b, mask_b = X_b.to(device), y_b.to(device), mask_b.to(device)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = masked_mse_loss(pred, y_b, mask_b)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_b, y_b, mask_b in val_loader:
            X_b, y_b, mask_b = X_b.to(device), y_b.to(device), mask_b.to(device)
            pred = model(X_b)
            loss = masked_mse_loss(pred, y_b, mask_b)
            epoch_val_loss += loss.item()

    avg_train = epoch_train_loss / len(train_loader)
    avg_val = epoch_val_loss / len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), CHECKPOINT_NAME)

    if epoch % 5 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Best: {best_val_loss:.4f}")

print(f"\nFinal best val loss: {best_val_loss:.4f} | RMSE: {math.sqrt(best_val_loss):.4f} °C")
print(f"Climatology baseline RMSE was: {math.sqrt(clim_mse):.4f} °C — model should beat this")

Epoch 1/50 | Train: 152.8404 | Val: 38.2238 | Best: 38.2238
Epoch 6/50 | Train: 2.1141 | Val: 2.2168 | Best: 2.0444
Epoch 11/50 | Train: 1.7758 | Val: 2.0470 | Best: 2.0444
Epoch 16/50 | Train: 1.6708 | Val: 3.1093 | Best: 2.0167
Epoch 21/50 | Train: 1.7176 | Val: 2.5707 | Best: 2.0167
Epoch 26/50 | Train: 1.3584 | Val: 2.6404 | Best: 2.0167
Epoch 31/50 | Train: 1.5275 | Val: 3.0533 | Best: 2.0167
Epoch 36/50 | Train: 2.0364 | Val: 2.5457 | Best: 2.0167
Epoch 41/50 | Train: 1.2397 | Val: 2.6689 | Best: 2.0167
Epoch 46/50 | Train: 1.1951 | Val: 4.5665 | Best: 2.0167
Epoch 50/50 | Train: 1.9124 | Val: 3.1375 | Best: 2.0167

Final best val loss: 2.0167 | RMSE: 1.4201 °C
Climatology baseline RMSE was: 1.6194 °C — model should beat this


In [37]:
import numpy as np
import torch

# ----------------------------
# Load clean checkpoint
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load("baseline_maskfix_v1.pth", map_location=device))
model.eval()

# ----------------------------
# Generate predictions
# ----------------------------
all_preds = []

with torch.no_grad():
    for X_b, y_b, mask_b in val_loader:
        X_b = X_b.to(device)
        pred = model(X_b).cpu().numpy()
        all_preds.append(pred)

preds_clean = np.concatenate(all_preds, axis=0).transpose(0,2,3,1)

print("Shape:", preds_clean.shape)
print("Min :", np.nanmin(preds_clean))
print("Max :", np.nanmax(preds_clean))
print("Mean:", np.nanmean(preds_clean))

# ----------------------------
# Save NEW predictions
# ----------------------------
np.save("val_predictions_maskfix_v1.npy", preds_clean)

print("\nSaved as val_predictions_maskfix_v1.npy")

Shape: (37, 76, 76, 14)
Min : -1.8845029
Max : 35.843338
Mean: 16.099236

Saved as val_predictions_maskfix_v1.npy


In [38]:
# Check prediction statistics over OCEAN cells only

ocean_mask = np.load("ocean_mask.npy").astype(bool)

ocean_preds = preds_clean[:, ocean_mask, :]

print("Ocean-only prediction statistics:")
print("Min :", np.min(ocean_preds))
print("Max :", np.max(ocean_preds))
print("Mean:", np.mean(ocean_preds))
print("Std :", np.std(ocean_preds))

print("\nPer-depth statistics:")
for i, depth in enumerate(
    [0.5, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700]
):
    vals = ocean_preds[:, :, i]
    print(
        f"{depth:>5} m : "
        f"min={np.min(vals):7.2f}, "
        f"max={np.max(vals):7.2f}, "
        f"mean={np.mean(vals):7.2f}"
    )

Ocean-only prediction statistics:
Min : -0.88812745
Max : 35.12013
Mean: 22.101212
Std : 7.8126254

Per-depth statistics:
  0.5 m : min=  24.96, max=  33.87, mean=  30.16
    5 m : min=  25.43, max=  33.50, mean=  30.02
   10 m : min=  23.73, max=  33.71, mean=  29.92
   20 m : min=  24.01, max=  35.12, mean=  29.74
   30 m : min=  18.88, max=  31.11, mean=  29.46
   50 m : min=  20.33, max=  31.12, mean=  28.18
   75 m : min=  14.24, max=  28.82, mean=  25.73
  100 m : min=  18.19, max=  25.77, mean=  22.88
  125 m : min=   3.62, max=  23.49, mean=  19.74
  150 m : min=  12.43, max=  19.68, mean=  17.71
  200 m : min=  -0.89, max=  17.23, mean=  14.33
  300 m : min=   5.93, max=  14.62, mean=  12.18
  500 m : min=   2.43, max=  11.52, mean=  10.19
  700 m : min=   5.96, max=  14.54, mean=   9.19


In [39]:
y_val = np.load("y_val.npy")

# Valid target mask
target_valid_val = np.isfinite(y_val)

print("Depth-wise validation statistics:\n")

depths = [0.5, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700]

for i, depth in enumerate(depths):
    valid = target_valid_val[:, :, :, i]

    true_vals = y_val[:, :, :, i][valid]
    pred_vals = preds_clean[:, :, :, i][valid]

    rmse = np.sqrt(np.mean((pred_vals - true_vals) ** 2))
    mae = np.mean(np.abs(pred_vals - true_vals))

    print(
        f"{depth:>5} m : "
        f"True mean={np.mean(true_vals):6.2f} | "
        f"Pred mean={np.mean(pred_vals):6.2f} | "
        f"RMSE={rmse:6.3f} | "
        f"MAE={mae:6.3f}"
    )

Depth-wise validation statistics:

  0.5 m : True mean= 29.99 | Pred mean= 30.17 | RMSE= 0.872 | MAE= 0.672
    5 m : True mean= 29.95 | Pred mean= 30.02 | RMSE= 0.825 | MAE= 0.629
   10 m : True mean= 29.93 | Pred mean= 29.94 | RMSE= 0.606 | MAE= 0.453
   20 m : True mean= 29.91 | Pred mean= 29.81 | RMSE= 0.609 | MAE= 0.431
   30 m : True mean= 29.80 | Pred mean= 29.62 | RMSE= 0.747 | MAE= 0.498
   50 m : True mean= 28.64 | Pred mean= 28.42 | RMSE= 1.499 | MAE= 1.182
   75 m : True mean= 25.91 | Pred mean= 26.02 | RMSE= 2.103 | MAE= 1.663
  100 m : True mean= 22.82 | Pred mean= 23.08 | RMSE= 2.433 | MAE= 1.994
  125 m : True mean= 20.01 | Pred mean= 20.01 | RMSE= 2.522 | MAE= 2.078
  150 m : True mean= 17.67 | Pred mean= 17.83 | RMSE= 2.047 | MAE= 1.674
  200 m : True mean= 14.52 | Pred mean= 14.41 | RMSE= 1.257 | MAE= 0.988
  300 m : True mean= 11.90 | Pred mean= 12.26 | RMSE= 0.766 | MAE= 0.609
  500 m : True mean= 10.09 | Pred mean= 10.37 | RMSE= 0.458 | MAE= 0.374
  700 m : True m

In [40]:
model.load_state_dict(torch.load("baseline_maskfix_v1.pth"))

<All keys matched successfully>